In [1]:
import os
import torch
from equiv_dens.training.parse_command_line_arguments import parse_command_line_arguments
from equiv_dens.data.density_dataset import AtomsDensityData
from equiv_dens.utils.grids import cubical_grid, cubical_sampling,\
    spherical_grid, spherical_radial_sampling
import equiv_dens.utils.base as utils
from equiv_dens.training.model_loader import load_model

import numpy as np
from functools import partial
import argparse
import copy
import ase.io
%load_ext autoreload
%autoreload 2

Use "numpy" for Fourier Transform


/home/mihail/anaconda3/envs/equiv_dens/lib/python3.7/site-packages/pyscf/lib/misc.py:47: H5pyDeprecationWarning: Using default_file_mode other than 'r' is deprecated. Pass the mode to h5py.File() instead.
  h5py.get_config().default_file_mode = 'a'


In [ ]:
class LoadFromFile (argparse.Action):
    def __call__ (self, parser, namespace, values, option_string = None):
        with values as f:
            # parse arguments in the file and store them in the target namespace
            parser.parse_args(f.read().split(), namespace)

In [ ]:
water_mo = np.load('datasets/h2o_static_pyscf_dft.npy', allow_pickle=True)
water_data = np.load('datasets/h2o_overlap_static_centered.npy', allow_pickle=True).item()

In [ ]:
print(water_data.keys())

In [ ]:
print(water_data['atom_types'])
print(water_data['atom_numbers'])

In [ ]:
water_dimer_data = {}
positions = []
water_dimer_data['atom_types'] = water_data['atom_types'] +  water_data['atom_types']
water_dimer_data['atom_numbers'] = np.concatenate([water_data['atom_numbers'], water_data['atom_numbers']])
for d in [5, 4, 3, 2]:
    offset = np.array([d/2, 0, 0])
    offset = np.reshape(offset, (1, 3))
    for i in range(len(water_data['positions'])):
        for j in range(len(water_data['positions'])):     
            positions.append(np.concatenate([water_data['positions'][i] - offset, water_data['positions'][j] + offset], axis=0))
            

positions = np.array(positions)
print(positions.shape)
water_dimer_data['positions'] = positions
mol = utils.npy_to_ase(positions, water_dimer_data['atom_types'])
    
np.save('datasets/water_dimer_test.npy', water_dimer_data)

In [ ]:



ase.io.write('datasets/water_dimer_test.xyz', mol)

In [2]:
args, hyperparam_args = parse_command_line_arguments(arg_file='water_dyn_spherical_test.txt')
print('type dtype', type(args.dtype))
args.fix_arguments = True
print('args np dir', args.np_dataset)
# no restart directory specified
directory = args.restart  # load directory name
# load latest checkpoint
checkpoint_path = os.path.join(directory, 'checkpoints')  # checkpoint directory
checkpoint = torch.load(os.path.join(
    checkpoint_path, 'latest_checkpoint.pth'), map_location='cpu')
latest_checkpoint = checkpoint['step']
model_code = checkpoint['ID']  # load ID
step = checkpoint['step']
for arg in vars(checkpoint['args']):
    if args.fix_arguments:
        if arg in hyperparam_args:
            print('loading hyperparam arg', arg)
            setattr(args, arg, getattr(checkpoint['args'], arg))
    else:
        print('loading all arg', arg)
        setattr(args, arg, getattr(checkpoint['args'], arg))
restore = True

args.best_model_path = 'best_' + model_code + '.pth'
print('best_model_path', args.best_model_path)

print('model code:', model_code)
# determine whether GPU is used for training
print('args use gpu', args.use_gpu)
args.use_gpu = args.use_gpu and torch.cuda.is_available()

# load dataset(s)
print("loading density from" + str(args.dens_dataset) + "...")
print("loading atoms from" + args.np_dataset + "...")
args.use_gpu = False
if args.cube_grid:
    grid_origin = args.cube_origin
    grid_extent = np.array([args.cube_extent] * 3)
    grid_fn = partial(cubical_grid, nx=args.cube_size, ny=args.cube_size, nz=args.cube_size,
                      extent=grid_extent,
                      origin=np.array([grid_origin] * 3))
    sampling_fn = cubical_sampling
else:
    grid_fn = partial(spherical_grid, level=2)
    sampling_fn = partial(spherical_radial_sampling, rotate=False)
    grid_origin = 0
    grid_extent = None

dataset = AtomsDensityData(np_path=args.np_dataset, density_path=args.dens_dataset,
                           orbitals_path=args.orbitals_file,
                           density_n_samp=10000000000,
                           required_properties=['density'],
                           center_positions=False,
                           radial_coeffs_file=args.radial_coeffs_file,
                           dtype=args.dtype,
                           grid_fn=grid_fn,
                           sampling_fn=sampling_fn,
                           grid_extent=grid_extent,
                           grid_origin=grid_origin,
                           verbose=args.verbose)
torch.manual_seed(0)
np.random.seed(0)
args.restart = None

model = load_model(args, dataset)

type dtype <class 'torch.dtype'>
args np dir datasets/h2o_dynamic_centered.npy
loading hyperparam arg activation
loading hyperparam arg order
loading hyperparam arg mixing_order
loading hyperparam arg order_en
loading hyperparam arg mixing_order_en
loading hyperparam arg num_features
loading hyperparam arg num_basis_functions
loading hyperparam arg num_radial_components
loading hyperparam arg num_energy_features
loading hyperparam arg num_modules
loading hyperparam arg num_residual_pre_x
loading hyperparam arg num_residual_post_x
loading hyperparam arg num_residual_pre_vi
loading hyperparam arg num_residual_pre_vj
loading hyperparam arg num_residual_post_v
loading hyperparam arg num_residual_output
loading hyperparam arg num_energy_output
loading hyperparam arg basis_functions
loading hyperparam arg cutoff
loading hyperparam arg orthonormal_basis
loading hyperparam arg expansion_constraint
loading hyperparam arg integral_constraint
loading hyperparam arg integral_scale
loading hyperpar

In [4]:
water_dimer = np.load('datasets/water_dimer_test.npy', allow_pickle=True).item()
print('len water_dimer', len(water_dimer['positions']))
print(water_dimer['atom_numbers'].shape)
min_dims = np.min(water_dimer['positions'], axis=(0,1)) - 1
max_dims = np.max(water_dimer['positions'], axis=(0,1)) + 1
span = max_dims - min_dims
step_size = max(span)/50
grid_size = np.round(span/step_size).astype(np.int)
dim_coords = []
for i in range(len(min_dims)):
    dim_coords.append(np.linspace(min_dims[i], max_dims[i], grid_size[i]))
x, y, z = np.meshgrid(*dim_coords, indexing='ij')
grid = np.stack([x, y, z], axis=-1)[np.newaxis, :]
grid = torch.Tensor(grid).to(args.dtype)
grid_weights = np.ones(grid.shape[:-1])
grid_weights /= np.sum(grid_weights)
grid_weights = torch.Tensor(grid_weights).to(args.dtype)

len water_dimer 1296
(6,)


In [ ]:
#print(min_dims)
#print(max_dims)
#print(span)
#print(step_size)
#print(grid_size)
#print(dim_coords)
#print(grid.shape)
#print(grid_weights.shape)
results = []
#for i in range(len(water_dimer['positions'])):
for i in np.linspace(0, 1295, 12).astype(np.int):
    print('working on water molecule', i)
    dimer_sample = {}
    result = {}
    dimer_sample['positions'] = torch.Tensor(water_dimer['positions'][[i]]).to(args.dtype)
    dimer_sample['atom_numbers'] = torch.LongTensor(water_dimer['atom_numbers']).unsqueeze(0)
    dimer_sample['coords'] = grid
    dimer_sample['coord_weights'] = grid_weights
    print('positions', dimer_sample['positions'])
    print('atom_numbers', dimer_sample['atom_numbers'])
    result = model(dimer_sample)
    print('energy', result['energy'])
    
    results.append(result)

In [3]:
from equiv_dens.nn.modules.embeddings import SphericalEmbedding

transfer_args = copy.deepcopy(args)

transfer_args.transferable_model = True
transfer_args.restart=None

torch.manual_seed(0)
np.random.seed(0)

transfer_model = load_model(transfer_args, dataset)

cg_matrix shape torch.Size([121, 121, 121])
args energy_unit_in hartree
args energy_unit_out kcal
self order [1, 1, 5]
self order [1, 1, 5]
self mixing_order [1, 1, 5]
self mixing_order [1, 1, 5]
Order needs to be an integer or a list of integers with length equal to num_modules. Taking last order element and using it for all modules.
Mixing order needs to be an integer or a list of integers with length equal to num_modules. Taking last order element and using it for all modules.
creating embedding
init_coeffs None
orbital basis {8: [(8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 2), (8, 1, 2), (8, 1, 2), (8, 1, 2), (8, 1, 2), (8, 1, 2), (8, 1, 3), (8, 1, 3), (8, 1, 3), (8, 1, 3), (8, 1, 4), (8, 1, 4), (8, 1, 4), (8, 1, 5), (8, 1, 5)], 1: [(1, 1, 0), (1, 1, 0), (1, 1, 0), (1, 1, 0), (1, 1, 0), (1, 1, 1), (1, 1, 1), (1, 

In [ ]:
for i in np.linspace(0, 1295, 12).astype(np.int):
    dimer_sample = {}
    result = {}
    dimer_sample['positions'] = torch.Tensor(water_dimer['positions'][[i]]).to(args.dtype)
    dimer_sample['atom_numbers'] = torch.LongTensor(water_dimer['atom_numbers']).unsqueeze(0)
    dimer_sample['atom_mask'] = dimer_sample['atom_numbers'] != 0
    dimer_sample['coords'] = grid
    dimer_sample['coord_weights'] = grid_weights
    dimer_sample_trans = copy.deepcopy(dimer_sample)
    print('positions', dimer_sample['positions'])
    print('atom_numbers', dimer_sample['atom_numbers'])
    transfer_repr = transfer_model.density_repr_model(dimer_sample_trans)
    repr = model.density_repr_model(dimer_sample)

    print('transfer model sph repr', transfer_repr['sph_repr'][0][..., :5])
    print('old model sph repr', repr['sph_repr'][0][..., :5])
#print(transfer_model.density_repr_model[0])

In [6]:
dimer_sample = {}
result = {}
dimer_sample['positions'] = torch.Tensor(water_dimer['positions'][[1200]]).to(args.dtype)
dimer_sample['atom_numbers'] = torch.LongTensor(water_dimer['atom_numbers']).unsqueeze(0)
dimer_sample['atom_mask'] = dimer_sample['atom_numbers'] != 0
print(water_dimer['atom_numbers'])
dimer_sample['coords'] = grid
dimer_sample['coord_weights'] = grid_weights
dimer_sample_trans = copy.deepcopy(dimer_sample)
print('positions', dimer_sample['positions'])
print('atom_numbers', dimer_sample['atom_numbers'])
transfer_repr = transfer_model(dimer_sample_trans)
repr = model(dimer_sample)

print('transfer model sph repr', transfer_repr['sph_repr'][0][..., :5])
print('old model sph repr', repr['sph_repr'][0][..., :5])
print('transfer model energy', transfer_repr['energy'])
print('old model energy', repr['energy'])
print('transfer model forces', transfer_repr['forces'])
print('old model forces', repr['forces'])
print('transfer model density shape', transfer_repr['density'][:, 25, 9,11])
print('old model density shape', repr['density'][:, 25, 9,11])
#print(transfer_model.density_repr_model[0])

[8 1 1 8 1 1]
positions tensor([[[-1.0000,  0.0000,  0.0000],
         [-0.0307, -0.1022,  0.0000],
         [-1.1022,  0.9693,  0.0000],
         [ 1.0000,  0.0000,  0.0000],
         [ 1.9693, -0.1022,  0.0000],
         [ 0.8978,  0.9693,  0.0000]]])
atom_numbers tensor([[8, 1, 1, 8, 1, 1]])
atom numbers tensor([[8, 1, 1, 8, 1, 1]])
atom mask tensor([[True, True, True, True, True, True]])
idx_i tensor([0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 4, 4, 4, 4,
        4, 5, 5, 5, 5, 5])
idx_j tensor([1, 2, 3, 4, 5, 0, 2, 3, 4, 5, 0, 1, 3, 4, 5, 0, 1, 2, 4, 5, 0, 1, 2, 3,
        5, 0, 1, 2, 3, 4])
neighbor_mask tensor([[True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True]])
self.order 5
self.orbitals_max_order 5
len fs 6
len out_sph 6
len out_width 6
len out_scale 6
extracting coefficients
spherical_spec {8: [(8, 11, 0), (8, 8,

In [7]:
dimer_sample = {}
dimer_sample['positions'] = torch.Tensor(water_dimer['positions'][[1200],:3]).to(args.dtype)
dimer_sample['atom_numbers'] = torch.LongTensor(water_dimer['atom_numbers'][:3]).unsqueeze(0)
dimer_sample['atom_mask'] = dimer_sample['atom_numbers'] != 0
dimer_sample['coords'] = grid
dimer_sample['coord_weights'] = grid_weights
dimer_sample_trans = copy.deepcopy(dimer_sample)

print('positions', dimer_sample['positions'])
print('atom_numbers', dimer_sample['atom_numbers'])
transfer_repr = transfer_model(dimer_sample_trans)
repr = model(dimer_sample)

print('transfer model sph repr', transfer_repr['sph_repr'][0][..., :5])
print('old model sph repr', repr['sph_repr'][0][..., :5])
print('transfer model energy', transfer_repr['energy'])
print('old model energy', repr['energy'])
print('transfer model forces', transfer_repr['forces'])
print('old model forces', repr['forces'])
print('transfer model density shape', transfer_repr['density'][:, 25, 9,11])
print('old model density shape', repr['density'][:, 25, 9,11])

positions tensor([[[-1.0000,  0.0000,  0.0000],
         [-0.0307, -0.1022,  0.0000],
         [-1.1022,  0.9693,  0.0000]]])
atom_numbers tensor([[8, 1, 1]])
atom numbers tensor([[8, 1, 1]])
atom mask tensor([[True, True, True]])
idx_i tensor([0, 0, 1, 1, 2, 2])
idx_j tensor([1, 2, 0, 2, 0, 1])
neighbor_mask tensor([[True, True, True, True, True, True]])
self.order 5
self.orbitals_max_order 5
len fs 6
len out_sph 6
len out_width 6
len out_scale 6
extracting coefficients
spherical_spec {8: [(8, 11, 0), (8, 8, 1), (8, 6, 2), (8, 4, 3), (8, 3, 4), (8, 2, 5)], 1: [(1, 5, 0), (1, 4, 1), (1, 4, 2), (1, 3, 3), (1, 2, 4)]}
radial_count {8: [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1], [1, 1, 1, 1], [1, 1, 1], [1, 1], []], 1: [[1, 1, 1, 1, 1], [1, 1, 1, 1], [1, 1, 1, 1], [1, 1, 1], [1, 1], [], []]}
forces create graph True
num grid points torch.Size([1, 50, 17, 22, 3])
Density shape torch.Size([1, 50, 17, 22])
atomic energies tensor([[-0.7662, -0.1011, -0.10

In [8]:
dimer_sample = {}
result = {}
dimer_sample['positions'] = torch.Tensor(water_dimer['positions'][[1200]]).to(args.dtype)
dimer_sample['atom_numbers'] = torch.tensor(water_dimer['atom_numbers']).type(torch.long).unsqueeze(0)
dimer_sample['atom_numbers'][:, 3:] = 0
dimer_sample['atom_mask'] = dimer_sample['atom_numbers'] != 0
dimer_sample['coords'] = grid
dimer_sample['coord_weights'] = grid_weights
dimer_sample_trans = copy.deepcopy(dimer_sample)
print('positions', dimer_sample['positions'])
print('atom_numbers', dimer_sample['atom_numbers'])
transfer_repr = transfer_model(dimer_sample_trans)
repr = model(dimer_sample)

print('transfer model sph repr', transfer_repr['sph_repr'][1][..., :5])
print('old model sph repr', repr['sph_repr'][1][..., :5])
print('transfer model energy', transfer_repr['energy'])
print('old model energy', repr['energy'])
print('transfer model forces', transfer_repr['forces'])
print('old model forces', repr['forces'])
print('transfer model density shape', transfer_repr['density'][:, 25, 9,11])
print('old model density shape', repr['density'][:, 25, 9,11])
#print(transfer_model.density_repr_model[0])

positions tensor([[[-1.0000,  0.0000,  0.0000],
         [-0.0307, -0.1022,  0.0000],
         [-1.1022,  0.9693,  0.0000],
         [ 1.0000,  0.0000,  0.0000],
         [ 1.9693, -0.1022,  0.0000],
         [ 0.8978,  0.9693,  0.0000]]])
atom_numbers tensor([[8, 1, 1, 0, 0, 0]])
atom numbers tensor([[8, 1, 1, 0, 0, 0]])
atom mask tensor([[ True,  True,  True, False, False, False]])
idx_i tensor([0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 4, 4, 4, 4,
        4, 5, 5, 5, 5, 5])
idx_j tensor([1, 2, 3, 4, 5, 0, 2, 3, 4, 5, 0, 1, 3, 4, 5, 0, 1, 2, 4, 5, 0, 1, 2, 3,
        5, 0, 1, 2, 3, 4])
neighbor_mask tensor([[ True,  True, False, False, False,  True,  True, False, False, False,
          True,  True, False, False, False,  True,  True,  True, False, False,
          True,  True,  True, False, False,  True,  True,  True, False, False]])
self.order 5
self.orbitals_max_order 5
len fs 6
len out_sph 6
len out_width 6
len out_scale 6
extracting coefficients
spherical_spec {